In [1]:
import pandas as pd
from generate_xml import load_data, write_xml
import re

In [2]:
data,metadata = load_data('../../zaebuc_written/ZAEBUC-v2.0_release/')
data.head()

/Users/f/Library/CloudStorage/SynologyDrive-ba3sasah/camelLab/zaebuc/corpus_app/src/generate_xml.py:7: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  en = pd.read_csv(f'{datadir}corrected.analyzed_en.tsv',sep='\t',index_col=[0,2,1])


word flag auto_tokenization auto_pos  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    Developments  NaN      Developments     NOUN   
                          2              in  NaN                in      ADP   
                          3             the  NaN               the      DET   
                          4             UAE  NaN               UAE    PROPN   
                          5               ,  NaN                 ,    PUNCT   

                                auto_lemma manual_tokenization manual_pos  \
doc_id         Line_Index idx                                               
en-2019-116710 1.0        1    development        Developments       NOUN   
                          2             in                  in        ADP   
                          3            the                 the        DET   
                          4            UAE                 UAE      PROPN   
                          5              ,                   ,      PUNCT   

                              manual_lemma comment manual_diacritized_lemma  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    development     NaN                      NaN   
                          2             in     NaN                      NaN   
                          3            the     NaN                      NaN   
                          4            UAE     NaN                      NaN   
                          5              ,     NaN                      NaN   

                              gloss core_pgn pron_pgn  
doc_id         Line_Index idx                          
en-2019-116710 1.0        1     NaN      NaN      NaN  
                          2     NaN      NaN      NaN  
                          3     NaN      NaN      NaN  
                          4     NaN      NaN      NaN  
                          5     NaN      NaN      NaN

# gloss expansion

In [3]:
import numpy as np
from copy import deepcopy 
# get arabic mask
isar = data.index.map(lambda x: 'ar' in x[0])

# split available glosses and creaate en gloss_search to ar manual_lemma map
data.loc[isar,'gloss_search'] = data['gloss'].map(lambda x: [x.strip().lower() for x in re.split(r',|;',x)],na_action='ignore')
glossearch = data.reset_index()[['manual_lemma','gloss_search']].explode('gloss_search').replace('',np.nan).drop_duplicates()
gloss_map = glossearch.dropna().groupby('gloss_search')['manual_lemma'].agg(list).to_dict()

# map english lemmas to arabic lemmas when english lemma is in gloss of arabic lemma
data.loc[~isar,'gloss_search'] = data.loc[~isar,'manual_lemma'].map(lambda x: deepcopy(gloss_map.get(x.lower(),[])))


# all glosses include original entry
# unique_gloss_lemma = data.copy().apply(lambda x: (tuple(x['gloss_search']),x['manual_lemma']),axis=1,result_type='expand').drop_duplicates(subset=[0,1]).index
# data.loc[unique_gloss_lemma].apply(lambda x: x['gloss_search'].append(x['manual_lemma']), axis=1)
data.apply(lambda x: x['gloss_search'].append(x['manual_lemma']), axis=1)
print()

In [4]:
data['gloss_search'][0]

/var/folders/nb/rn_wr53j4mbcyl_kw5fw8t3m0000gn/T/ipykernel_93892/2416000612.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  data['gloss_search'][0]


['تطوير', 'تنمية', 'تطور', 'نمو', 'نشأة', 'development']

In [5]:
data.head(2)

word flag auto_tokenization auto_pos  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    Developments  NaN      Developments     NOUN   
                          2              in  NaN                in      ADP   

                                auto_lemma manual_tokenization manual_pos  \
doc_id         Line_Index idx                                               
en-2019-116710 1.0        1    development        Developments       NOUN   
                          2             in                  in        ADP   

                              manual_lemma comment manual_diacritized_lemma  \
doc_id         Line_Index idx                                                 
en-2019-116710 1.0        1    development     NaN                      NaN   
                          2             in     NaN                      NaN   

                              gloss core_pgn pron_pgn  \
doc_id         Line_Index idx                           
en-2019-116710 1.0        1     NaN      NaN      NaN   
                          2     NaN      NaN      NaN   

                                                               gloss_search  
doc_id         Line_Index idx                                                
en-2019-116710 1.0        1    [تطوير, تنمية, تطور, نمو, نشأة, development]  
                          2                               [في, منذ, إن, in]

# write xml

In [6]:
write_xml(data, metadata, out_path="../data/zaebuc_written.xml", sample=None)
write_xml(data, metadata, out_path="../data/zaebuc_written_sample.xml", sample=100)

In [28]:
list(data.columns)

['word',
 'flag',
 'auto_tokenization',
 'auto_pos',
 'auto_lemma',
 'manual_tokenization',
 'manual_pos',
 'manual_lemma',
 'comment',
 'manual_diacritized_lemma',
 'gloss',
 'core_pgn',
 'pron_pgn',
 'gloss_search']